In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

## Wage levels (ai x ict_spec)

In [32]:
import matplotlib.pyplot as plt
import seaborn as sns

path = 'C:/Users/ydmar/Documents/UW/Master thesis/Master_thesis/data_panel/'

df_master = pd.read_csv(path + 'full_panel_master.csv')
print(df_master)

    geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0    AT       C      26.74  2021        9.610     28.56         22.98   
1    AT       C      26.18  2022       10.960     28.37         25.60   
2    AT       C      26.07  2023       12.310     29.20         25.83   
3    AT       C      27.05  2024       22.710     30.02         26.06   
4    AT       F      22.43  2021        3.120      8.64          8.28   
..   ..     ...        ...   ...          ...       ...           ...   
531  SK       G       8.45  2024       11.650     15.19         19.87   
532  SK       J      15.69  2021       18.190     70.76         55.96   
533  SK       J      14.79  2022       19.905     71.88         53.54   
534  SK       J      14.48  2023       21.620     70.80         58.16   
535  SK       J      14.67  2024       29.190     69.72         62.79   

     productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
0          102.25  55.82             36.58      3

## Balanced panel

In [33]:
df_master = pd.read_csv(path + 'panel_master.csv')
print(df_master)

    geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0    AT       C      26.74  2021        9.610     28.56         22.98   
1    AT       C      26.18  2022       10.960     28.37         25.60   
2    AT       C      26.07  2023       12.310     29.20         25.83   
3    AT       C      27.05  2024       22.710     30.02         26.06   
4    AT       F      22.43  2021        3.120      8.64          8.28   
..   ..     ...        ...   ...          ...       ...           ...   
531  SK       G       8.45  2024       11.650     15.19         19.87   
532  SK       J      15.69  2021       18.190     70.76         55.96   
533  SK       J      14.79  2022       19.905     71.88         53.54   
534  SK       J      14.48  2023       21.620     70.80         58.16   
535  SK       J      14.67  2024       29.190     69.72         62.79   

     productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
0          102.25  55.82             36.58      3

In [34]:
df_master['id'] = df_master['geo'] + '_' + df_master['nace_r2']
df_model = df_master.set_index(['id', 'year']).sort_index()

df_model['log_real_wage'] = np.log(df_model['real_wage'])
df_model['ai_x_ict'] = df_model['ai_adoption'] * df_model['spec_ict']


print(f"Observations: {len(df_model)}")
print(df_model.head())

Observations: 536
          geo nace_r2  real_wage  ai_adoption  spec_ict  training_ict  \
id   year                                                               
AT_C 2021  AT       C      26.74         9.61     28.56         22.98   
     2022  AT       C      26.18        10.96     28.37         25.60   
     2023  AT       C      26.07        12.31     29.20         25.83   
     2024  AT       C      27.05        22.71     30.02         26.06   
AT_F 2021  AT       F      22.43         3.12      8.64          8.28   

           productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
id   year                                                                      
AT_C 2021        102.25  55.82             36.58      31.8  111.46      4.63   
     2022        108.40  56.10             37.32      33.0  121.07      4.69   
     2023        108.93  57.52             38.71      33.5  130.40      4.69   
     2024        108.60  57.66             40.13      35.8  134.21    

In [4]:
# entity_effects=True - cross-sectional effect 
# time_effects=True - time fixed effect 
# without these variables will be just pooled OLS
# cluster_entity=True - calculates Clustered Standard Errors (treat the grouped observations (Austria-sector C) as a single cluster (group), not independently). Allow the errors within this group to be correlated

In [35]:
y = df_model['log_real_wage']

exog_vars = [
    'ai_adoption',       # Main Effect: AI
    'spec_ict',          # Main Effect: ICT Specialists
    'ai_x_ict',          # Interaction: AI * ICT
    # 'training_ict',      # Control: ICT Training
    'log_prod',      # Control: Productivity (t-1)
    'FSI',               # Control: Firm Size
    'share_high_skill'   # Control: Human Capital
]

X = df_model[exog_vars]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_panel_full = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_full)


                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.2265
Estimator:                   PanelOLS   R-squared (Between):              0.3570
No. Observations:                 536   R-squared (Within):              -0.0456
Date:                Tue, Feb 24 2026   R-squared (Overall):              0.3546
Time:                        00:19:56   Log-likelihood                    1035.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      19.184
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,393)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             5.2950
                            

In [ ]:
X = df_model[exog_vars]
X = sm.add_constant(X)

mod = PanelOLS(y, X, entity_effects=True)
res_panel_cs_FE = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_cs_FE)

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.0868
Estimator:                   PanelOLS   R-squared (Between):              0.0821
No. Observations:                 536   R-squared (Within):               0.0868
Date:                Mon, Feb 23 2026   R-squared (Overall):              0.0821
Time:                        22:47:48   Log-likelihood                    923.38
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      6.2728
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             4.2380
                            

In [8]:
from linearmodels.panel import PooledOLS

# Pooled OLS does not use Entity Effects
# It treats every row as an independent observation
mod_pool = PooledOLS(y, X)
res_pool = mod_pool.fit(cov_type='clustered', cluster_entity=True)
print(res_pool)

                          PooledOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.7552
Estimator:                  PooledOLS   R-squared (Between):              0.7974
No. Observations:                 536   R-squared (Within):              -6.0826
Date:                Mon, Feb 23 2026   R-squared (Overall):              0.7552
Time:                        22:48:11   Log-likelihood                   -89.075
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      272.05
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             84.954
                            

In [9]:
from linearmodels.panel import BetweenOLS, FirstDifferenceOLS, RandomEffects
import pandas as pd
import statsmodels.api as sm

mod_be = BetweenOLS(y, X)
res_be = mod_be.fit()
print(res_be)

                         BetweenOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.8186
Estimator:                 BetweenOLS   R-squared (Between):              0.8186
No. Observations:                 134   R-squared (Within):              -16.136
Date:                Mon, Feb 23 2026   R-squared (Overall):              0.7147
Time:                        22:48:34   Log-likelihood                   -1.7746
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      95.537
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,127)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             95.537
                            

In [10]:
# "Weighted average of Fixed Effects and Between Effects."
# Best model if your unobserved country traits are NOT correlated with AI.
mod_re = RandomEffects(y, X)
res_re = mod_re.fit()
print(res_re)


                        RandomEffects Estimation Summary                        
Dep. Variable:          log_real_wage   R-squared:                        0.2457
Estimator:              RandomEffects   R-squared (Between):              0.4530
No. Observations:                 536   R-squared (Within):              -0.0891
Date:                Mon, Feb 23 2026   R-squared (Overall):              0.4497
Time:                        22:48:39   Log-likelihood                    717.01
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      28.718
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             28.718
                            

In [11]:
# Similar to Fixed Effects, often handles trends better.
X_fd = X.drop(columns=['const'])
mod_fd = FirstDifferenceOLS(y, X_fd)
res_fd = mod_fd.fit()
print(res_fd)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:          log_real_wage   R-squared:                        0.1081
Estimator:         FirstDifferenceOLS   R-squared (Between):             -0.0163
No. Observations:                 402   R-squared (Within):               0.0380
Date:                Mon, Feb 23 2026   R-squared (Overall):             -0.0163
Time:                        22:48:43   Log-likelihood                    615.56
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      8.0028
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             8.0028
                            

In [12]:
from linearmodels.panel import compare

comparison = {
    'Pooled OLS': res_pool,
    'Between': res_be,
    'Random Effects': res_re,
    'Fixed Effects full': res_panel_full, 
    'Fixed Effect cross-sect' : res_panel_cs_FE,
    'First Diff': res_fd
}

summary_table = compare(comparison)

print(summary_table)

                                                                Model Comparison                                                               
                               Pooled OLS           Between    Random Effects Fixed Effects full Fixed Effect cross-sect             First Diff
-----------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable               log_real_wage     log_real_wage     log_real_wage      log_real_wage           log_real_wage          log_real_wage
Estimator                       PooledOLS        BetweenOLS     RandomEffects           PanelOLS                PanelOLS     FirstDifferenceOLS
No. Observations                      536               134               536                536                     536                    402
Cov. Est.                       Clustered        Unadjusted        Unadjusted          Clustered               Clustered             Una

In [13]:
import statsmodels.stats.diagnostic as smd
from scipy import stats

# Model selection test (WALD TEST)
# https://bashtage.github.io/linearmodels/panel/panel/linearmodels.panel.results.PanelEffectsResults.f_pooled.html
# 1. F-Test for Fixed Effects (Pooled OLS vs. Fixed Effects)
# H0: Pooled OLS is better (Entity effects are zero)
# H1: Fixed Effects is bette

print(f"1. F-Test for Entity Effects (Pooled vs FE):")
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")



1. F-Test for Entity Effects (Pooled vs FE):
   F-Stat: 188.8057
   P-Value: 0.0000
   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.


In [14]:
# 2. Breusch-Pagan LM Test (Pooled OLS vs. Random Effects)
# H0: Variance of random effects is 0 (Pooled OLS is fine)
# H1: Variance > 0 (Random Effects are needed)
# We calculate this manually using residuals from Pooled OLS
resid_pool = res_pool.resids
n = len(df_model.index.get_level_values(0).unique()) # Entities
T = len(df_model.index.get_level_values(1).unique()) # Time periods
# Calculation
lm_stat = (n * T) / (2 * (T - 1)) * (
    (resid_pool.groupby(level=0).sum() ** 2).sum() / (resid_pool ** 2).sum() - 1
) ** 2
lm_pval = 1 - stats.chi2.cdf(lm_stat, df=1)

print(f"\n2. Breusch-Pagan LM Test (Pooled vs Random Effects):")
print(f"   LM Stat: {lm_stat:.4f}")
print(f"   P-Value: {lm_pval:.4f}")
if lm_pval < 0.05:
    print("   -> Result: REJECT H0. Random Effects is better than Pooled.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")


2. Breusch-Pagan LM Test (Pooled vs Random Effects):
   LM Stat: 468.6424
   P-Value: 0.0000
   -> Result: REJECT H0. Random Effects is better than Pooled.


In [15]:
# 3. Hausman Test (Fixed Effects vs. Random Effects)
# H0: Random Effects is consistent (Use RE - it's more efficient)
# H1: Random Effects is biased (Use FE - it's safer)
b_fe = res_panel_full.params
b_re = res_re.params
cov_fe = res_panel_full.cov
cov_re = res_re.cov
# Calculate Chi-Square
diff = b_fe - b_re
# Note: Usually we drop the constant for Hausman as FE doesn't estimate it the same way
diff = diff.drop('const')
cov_diff = cov_fe.loc[diff.index, diff.index] - cov_re.loc[diff.index, diff.index]
hausman_stat = diff.dot(np.linalg.inv(cov_diff)).dot(diff)
hausman_pval = 1 - stats.chi2.cdf(hausman_stat, df=len(diff))

print(f"\n3. Hausman Test (FE vs RE):")
print(f"   Chi2 Stat: {hausman_stat:.4f}")
print(f"   P-Value: {hausman_pval:.4f}")
if hausman_pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects (RE is biased).")
else:
    print("   -> Result: ACCEPT H0. Use Random Effects (It is safe & efficient).")


3. Hausman Test (FE vs RE):
   Chi2 Stat: 147.0239
   P-Value: 0.0000
   -> Result: REJECT H0. Use Fixed Effects (RE is biased).


In [17]:
# 4. Heteroskedasticity Tests (White & Breusch-Pagan)
# We run these on the POOLED residuals (standard approach)
# H0: Homoskedasticity (Variance is constant) -> Good
# H1: Heteroskedasticity (Variance changes) -> Bad (Need Robust Errors)
bp_test = smd.het_breuschpagan(res_panel_full.resids, res_panel_full.model.exog.dataframe)
white_test = smd.het_white(res_panel_full.resids, res_panel_full.model.exog.dataframe)

print(f"4. Heteroskedasticity Tests:")
print(f"   Breusch-Pagan P-Value: {bp_test[1]:.4f}")
print(f"   White Test P-Value:    {white_test[1]:.4f}")
if bp_test[1] < 0.05:
    print("   -> Result: HETEROSKEDASTICITY DETECTED. (we must use cov_type='clustered')")
else:
    print("   -> Result: Homoskedasticity. (Data is clean).")

4. Heteroskedasticity Tests:
   Breusch-Pagan P-Value: 0.0130
   White Test P-Value:    0.0000
   -> Result: HETEROSKEDASTICITY DETECTED. (we must use cov_type='clustered')


In [ ]:
# # 5. Serial Correlation (Breusch-Godfrey)
# # H0: No Serial Correlation
# # H1: Serial Correlation exists
# # Note: We restrict lags to 1 because you only have 3 years of data
# bg_test = smd.acorr_breusch_godfrey(res_panel_full, nlags=1)

# print(f"\n5. Serial Correlation (Breusch-Godfrey):")
# print(f"   P-Value: {bg_test[1]:.4f}")
# if bg_test[1] < 0.05:
#     print("   -> Result: SERIAL CORRELATION DETECTED. (Use Clustered Errors).")
# else:
#     print("   -> Result: No Serial Correlation.")

In [19]:
# 6. Chow Test (Test for Poolability)
# This is automatically calculated in the PanelOLS summary!
# It tests if the slopes are different for every entity.
# linearmodels calls this "F-test for Poolability"
print(f"6. Chow Test (Poolability):")
# We access the F-statistic directly from the results object
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).")


6. Chow Test (Poolability):
   F-Stat: 188.8057
   P-Value: 0.0000
   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).


In [20]:
# 7. Wald Test (Joint Significance)
# Example: Does 'training_ict' AND 'share_high_skill' jointly equal zero?
# Use this to test if your Control Variables matter as a group.
formula = 'training_ict = 0, share_high_skill = 0'
# Note: If you removed these variables, change the formula to 'spec_ict = 0, ai_adoption = 0'
try:
    wald_res = res_panel_full.wald_test(formula=formula)
    print(f"\n7. Wald Test (Joint Significance of Controls):")
    print(f"   Stat: {wald_res.stat:.4f}, P-Value: {wald_res.pval:.4f}")
except:
    print("\n7. Wald Test: Skipped (Variables not in model).")


7. Wald Test: Skipped (Variables not in model).


In [21]:
# Calculate the variance
variance_data = df_model['ai_adoption']
mean_total = variance_data.mean()

# Between Variance (Variation across countries)
between_var = df_model.groupby('id')['ai_adoption'].mean().var()

# Within Variance (Variation over time within a country)
within_var = (df_model['ai_adoption'] - df_model.groupby('id')['ai_adoption'].transform('mean')).var()

print(f"Variance BETWEEN Sectors (Structure): {between_var:.4f}")
print(f"Variance WITHIN Sectors (Time):       {within_var:.4f}")

ratio = between_var / within_var
print(f"Ratio (Between / Within):             {ratio:.2f}")

Variance BETWEEN Sectors (Structure): 115.2127
Variance WITHIN Sectors (Time):       20.6323
Ratio (Between / Within):             5.58


## Wage levels (ai x ict_trainings)

In [37]:
df_master = pd.read_csv(path + 'panel_master.csv')
print(df_master)

    geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0    AT       C      26.74  2021        9.610     28.56         22.98   
1    AT       C      26.18  2022       10.960     28.37         25.60   
2    AT       C      26.07  2023       12.310     29.20         25.83   
3    AT       C      27.05  2024       22.710     30.02         26.06   
4    AT       F      22.43  2021        3.120      8.64          8.28   
..   ..     ...        ...   ...          ...       ...           ...   
531  SK       G       8.45  2024       11.650     15.19         19.87   
532  SK       J      15.69  2021       18.190     70.76         55.96   
533  SK       J      14.79  2022       19.905     71.88         53.54   
534  SK       J      14.48  2023       21.620     70.80         58.16   
535  SK       J      14.67  2024       29.190     69.72         62.79   

     productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
0          102.25  55.82             36.58      3

In [38]:

df_master['id'] = df_master['geo'] + '_' + df_master['nace_r2']
df_model = df_master.set_index(['id', 'year']).sort_index()
df_model['log_real_wage'] = np.log(df_model['real_wage'])
df_model['ai_x_ict_tr'] = df_model['ai_adoption'] * df_model['training_ict']

print(f"Observations: {len(df_model)}")
print(df_model.head())

Observations: 536
          geo nace_r2  real_wage  ai_adoption  spec_ict  training_ict  \
id   year                                                               
AT_C 2021  AT       C      26.74         9.61     28.56         22.98   
     2022  AT       C      26.18        10.96     28.37         25.60   
     2023  AT       C      26.07        12.31     29.20         25.83   
     2024  AT       C      27.05        22.71     30.02         26.06   
AT_F 2021  AT       F      22.43         3.12      8.64          8.28   

           productivity    FSI  share_high_skill  tert_edu    hicp  log_prod  \
id   year                                                                      
AT_C 2021        102.25  55.82             36.58      31.8  111.46      4.63   
     2022        108.40  56.10             37.32      33.0  121.07      4.69   
     2023        108.93  57.52             38.71      33.5  130.40      4.69   
     2024        108.60  57.66             40.13      35.8  134.21    

In [39]:
y = df_model['log_real_wage']
exog_vars = [
    'ai_adoption',       # Main Effect: AI
    # 'spec_ict',          # Main Effect: ICT Specialists
    'ai_x_ict_tr',          # Interaction: AI * ICT Training
    'training_ict',      # Control: ICT Training
    'log_prod',      # Control: Productivity (t-1)
    'FSI',               # Control: Firm Size
    'share_high_skill'   # Control: Human Capital
]
X = df_model[exog_vars]
X = sm.add_constant(X)
mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
res_panel_full = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_full)


                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.2461
Estimator:                   PanelOLS   R-squared (Between):              0.4233
No. Observations:                 536   R-squared (Within):              -0.0436
Date:                Tue, Feb 24 2026   R-squared (Overall):              0.4204
Time:                        00:40:22   Log-likelihood                    1042.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.381
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,393)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             7.1724
                            

In [40]:
X = df_model[exog_vars]
X = sm.add_constant(X)
mod = PanelOLS(y, X, entity_effects=True)
res_panel_cs_FE = mod.fit(cov_type='clustered', cluster_entity=True)
print(res_panel_cs_FE)

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_real_wage   R-squared:                        0.0971
Estimator:                   PanelOLS   R-squared (Between):              0.1944
No. Observations:                 536   R-squared (Within):               0.0971
Date:                Tue, Feb 24 2026   R-squared (Overall):              0.1938
Time:                        00:40:32   Log-likelihood                    926.41
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.0968
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             5.3135
                            

In [41]:
# Pooled OLS does not use Entity Effects
# It treats every row as an independent observation
mod_pool = PooledOLS(y, X)
res_pool = mod_pool.fit(cov_type='clustered', cluster_entity=True)
print(res_pool)

                          PooledOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.7426
Estimator:                  PooledOLS   R-squared (Between):              0.7752
No. Observations:                 536   R-squared (Within):              -4.5528
Date:                Tue, Feb 24 2026   R-squared (Overall):              0.7426
Time:                        00:41:46   Log-likelihood                   -102.60
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      254.32
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             44.578
                            

In [42]:
mod_be = BetweenOLS(y, X)
res_be = mod_be.fit()
print(res_be)

                         BetweenOLS Estimation Summary                          
Dep. Variable:          log_real_wage   R-squared:                        0.7902
Estimator:                 BetweenOLS   R-squared (Between):              0.7902
No. Observations:                 134   R-squared (Within):              -12.438
Date:                Tue, Feb 24 2026   R-squared (Overall):              0.7091
Time:                        00:42:11   Log-likelihood                   -11.531
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      79.722
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,127)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             79.722
                            

In [43]:
# "Weighted average of Fixed Effects and Between Effects."
# Best model if your unobserved country traits are NOT correlated with AI.
mod_re = RandomEffects(y, X)
res_re = mod_re.fit()
print(res_re)

                        RandomEffects Estimation Summary                        
Dep. Variable:          log_real_wage   R-squared:                        0.2405
Estimator:              RandomEffects   R-squared (Between):              0.4333
No. Observations:                 536   R-squared (Within):              -0.0256
Date:                Tue, Feb 24 2026   R-squared (Overall):              0.4305
Time:                        00:44:20   Log-likelihood                    740.36
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      27.916
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,529)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             27.916
                            

In [ ]:
# Similar to Fixed Effects, often handles trends better.
X_fd = X.drop(columns=['const'])
mod_fd = FirstDifferenceOLS(y, X_fd)
res_fd = mod_fd.fit()print(res_fd)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:          log_real_wage   R-squared:                        0.1139
Estimator:         FirstDifferenceOLS   R-squared (Between):              0.0724
No. Observations:                 402   R-squared (Within):               0.0511
Date:                Tue, Feb 24 2026   R-squared (Overall):              0.0724
Time:                        00:44:22   Log-likelihood                    616.86
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      8.4821
Entities:                         134   P-value                           0.0000
Avg Obs:                       4.0000   Distribution:                   F(6,396)
Min Obs:                       4.0000                                           
Max Obs:                       4.0000   F-statistic (robust):             8.4821
                            

In [45]:
comparison = {
    'Pooled OLS': res_pool,
    'Between': res_be,
    'Random Effects': res_re,
    'Fixed Effects full': res_panel_full, 
    'Fixed Effect cross-sect' : res_panel_cs_FE,
    'First Diff': res_fd
}
summary_table = compare(comparison)
print(summary_table)

                                                                Model Comparison                                                               
                               Pooled OLS           Between    Random Effects Fixed Effects full Fixed Effect cross-sect             First Diff
-----------------------------------------------------------------------------------------------------------------------------------------------
Dep. Variable               log_real_wage     log_real_wage     log_real_wage      log_real_wage           log_real_wage          log_real_wage
Estimator                       PooledOLS        BetweenOLS     RandomEffects           PanelOLS                PanelOLS     FirstDifferenceOLS
No. Observations                      536               134               536                536                     536                    402
Cov. Est.                       Clustered        Unadjusted        Unadjusted          Clustered               Clustered             Una

In [46]:
# Model selection test (WALD TEST)
# https://bashtage.github.io/linearmodels/panel/panel/linearmodels.panel.results.PanelEffectsResults.f_pooled.html
# 1. F-Test for Fixed Effects (Pooled OLS vs. Fixed Effects)
# H0: Pooled OLS is better (Entity effects are zero)
# H1: Fixed Effects is bette

print(f"1. F-Test for Entity Effects (Pooled vs FE):")
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")



1. F-Test for Entity Effects (Pooled vs FE):
   F-Stat: 203.9620
   P-Value: 0.0000
   -> Result: REJECT H0. Use Fixed Effects over Pooled OLS.


In [47]:
# 2. Breusch-Pagan LM Test (Pooled OLS vs. Random Effects)
# H0: Variance of random effects is 0 (Pooled OLS is fine)
# H1: Variance > 0 (Random Effects are needed)
# We calculate this manually using residuals from Pooled OLS
resid_pool = res_pool.resids
n = len(df_model.index.get_level_values(0).unique()) # Entities
T = len(df_model.index.get_level_values(1).unique()) # Time periods
# Calculation
lm_stat = (n * T) / (2 * (T - 1)) * (
    (resid_pool.groupby(level=0).sum() ** 2).sum() / (resid_pool ** 2).sum() - 1
) ** 2
lm_pval = 1 - stats.chi2.cdf(lm_stat, df=1)

print(f"\n2. Breusch-Pagan LM Test (Pooled vs Random Effects):")
print(f"   LM Stat: {lm_stat:.4f}")
print(f"   P-Value: {lm_pval:.4f}")
if lm_pval < 0.05:
    print("   -> Result: REJECT H0. Random Effects is better than Pooled.")
else:
    print("   -> Result: ACCEPT H0. Pooled OLS is sufficient.")


2. Breusch-Pagan LM Test (Pooled vs Random Effects):
   LM Stat: 545.4818
   P-Value: 0.0000
   -> Result: REJECT H0. Random Effects is better than Pooled.


In [48]:
# 3. Hausman Test (Fixed Effects vs. Random Effects)
# H0: Random Effects is consistent (Use RE - it's more efficient)
# H1: Random Effects is biased (Use FE - it's safer)
b_fe = res_panel_full.params
b_re = res_re.params
cov_fe = res_panel_full.cov
cov_re = res_re.cov
# Calculate Chi-Square
diff = b_fe - b_re
# Note: Usually we drop the constant for Hausman as FE doesn't estimate it the same way
diff = diff.drop('const')
cov_diff = cov_fe.loc[diff.index, diff.index] - cov_re.loc[diff.index, diff.index]
hausman_stat = diff.dot(np.linalg.inv(cov_diff)).dot(diff)
hausman_pval = 1 - stats.chi2.cdf(hausman_stat, df=len(diff))

print(f"\n3. Hausman Test (FE vs RE):")
print(f"   Chi2 Stat: {hausman_stat:.4f}")
print(f"   P-Value: {hausman_pval:.4f}")
if hausman_pval < 0.05:
    print("   -> Result: REJECT H0. Use Fixed Effects (RE is biased).")
else:
    print("   -> Result: ACCEPT H0. Use Random Effects (It is safe & efficient).")


3. Hausman Test (FE vs RE):
   Chi2 Stat: 7.2937
   P-Value: 0.2945
   -> Result: ACCEPT H0. Use Random Effects (It is safe & efficient).


In [49]:
# 4. Heteroskedasticity Tests (White & Breusch-Pagan)
# We run these on the POOLED residuals (standard approach)
# H0: Homoskedasticity (Variance is constant) -> Good
# H1: Heteroskedasticity (Variance changes) -> Bad (Need Robust Errors)
bp_test = smd.het_breuschpagan(res_panel_full.resids, res_panel_full.model.exog.dataframe)
white_test = smd.het_white(res_panel_full.resids, res_panel_full.model.exog.dataframe)

print(f"4. Heteroskedasticity Tests:")
print(f"   Breusch-Pagan P-Value: {bp_test[1]:.4f}")
print(f"   White Test P-Value:    {white_test[1]:.4f}")
if bp_test[1] < 0.05:
    print("   -> Result: HETEROSKEDASTICITY DETECTED. (we must use cov_type='clustered')")
else:
    print("   -> Result: Homoskedasticity. (Data is clean).")

4. Heteroskedasticity Tests:
   Breusch-Pagan P-Value: 0.0546
   White Test P-Value:    0.0000
   -> Result: Homoskedasticity. (Data is clean).


In [50]:
# 6. Chow Test (Test for Poolability)
# This is automatically calculated in the PanelOLS summary!
# It tests if the slopes are different for every entity.
# linearmodels calls this "F-test for Poolability"
print(f"6. Chow Test (Poolability):")
# We access the F-statistic directly from the results object
print(f"   F-Stat: {res_panel_full.f_pooled.stat:.4f}")
print(f"   P-Value: {res_panel_full.f_pooled.pval:.4f}")
if res_panel_full.f_pooled.pval < 0.05:
    print("   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).")


6. Chow Test (Poolability):
   F-Stat: 203.9620
   P-Value: 0.0000
   -> Result: Coefficients are NOT stable across entities. (Panel models are necessary).


In [51]:
# 7. Wald Test (Joint Significance)
# Example: Does 'training_ict' AND 'share_high_skill' jointly equal zero?
# Use this to test if your Control Variables matter as a group.
formula = 'training_ict = 0, share_high_skill = 0'
# Note: If you removed these variables, change the formula to 'spec_ict = 0, ai_adoption = 0'
try:
    wald_res = res_panel_full.wald_test(formula=formula)
    print(f"\n7. Wald Test (Joint Significance of Controls):")
    print(f"   Stat: {wald_res.stat:.4f}, P-Value: {wald_res.pval:.4f}")
except:
    print("\n7. Wald Test: Skipped (Variables not in model).")


7. Wald Test (Joint Significance of Controls):
   Stat: 10.1846, P-Value: 0.0061


In [52]:
# Calculate the variance
variance_data = df_model['ai_adoption']
mean_total = variance_data.mean()

# Between Variance (Variation across countries)
between_var = df_model.groupby('id')['ai_adoption'].mean().var()

# Within Variance (Variation over time within a country)
within_var = (df_model['ai_adoption'] - df_model.groupby('id')['ai_adoption'].transform('mean')).var()

print(f"Variance BETWEEN Sectors (Structure): {between_var:.4f}")
print(f"Variance WITHIN Sectors (Time):       {within_var:.4f}")

ratio = between_var / within_var
print(f"Ratio (Between / Within):             {ratio:.2f}")

Variance BETWEEN Sectors (Structure): 115.2127
Variance WITHIN Sectors (Time):       20.6323
Ratio (Between / Within):             5.58
